# 03. 하이퍼파라미터 튜닝 실험

베이스라인(#00)과 동일한 결측치 처리 + 파생변수 17개 + 인코딩(순서형 + LabelEncoder)을 사용하고,
`LGBMRegressor`의 하이퍼파라미터만 몇 가지 조합으로 바꿔서 CV MAE 변화를 비교합니다.

비교 기준: #00 (기본 하이퍼파라미터) CV MAE = 0.2117

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42  # 그라운드룰 1: 항상 42로 고정

## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('train:', train.shape, '/ test:', test.shape)

train: (3000, 18) / test: (3000, 17)


## 2. 결측치 및 중복행 처리 (0909 ver, 팀 확정본)

In [3]:
# 중복 행 제거 (ID 제외 기준) - train에만 적용, test는 제거하지 않음
train = train.drop_duplicates(
    subset=[col for col in train.columns if col not in ['ID']]
).reset_index(drop=True)

# 근로시간 결측치: 0 처리
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 범주형 결측치: 독립 범주 신설
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('결측치 처리 후 남은 결측 개수 - train:', train.isnull().sum().sum(), '/ test:', test.isnull().sum().sum())
print('중복 제거 후 train shape:', train.shape)

결측치 처리 후 남은 결측 개수 - train: 0 / test: 0
중복 제거 후 train shape: (2994, 18)


## 3. 파생변수 생성 (0909 ver, 팀 확정본 17개)

**중요**: 반드시 2단계(결측치 fillna)가 끝난 뒤, 4단계(인코딩) 이전에 실행해야 합니다.

In [4]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data

train = add_features(train)
test = add_features(test)

print('파생변수 추가 후 train shape:', train.shape)

파생변수 추가 후 train shape: (2994, 35)


## 4. 인코딩 (#00과 동일: Ordinal + LabelEncoder 혼합)

In [5]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)

x_train: (2994, 33) / x_test: (3000, 33)


## 5. 하이퍼파라미터 조합별 5-Fold CV 비교

아래 후보들을 바꿔가며 자유롭게 추가/수정해서 테스트해보세요.

In [6]:
param_candidates = {
    "baseline(#00)": dict(random_state=RANDOM_STATE),
    "more_trees_slow_lr": dict(random_state=RANDOM_STATE, n_estimators=1000, learning_rate=0.03),
    "deeper": dict(random_state=RANDOM_STATE, num_leaves=63, max_depth=8),
    "regularized": dict(random_state=RANDOM_STATE, n_estimators=800, learning_rate=0.05,
                         reg_alpha=0.1, reg_lambda=0.1, subsample=0.8, colsample_bytree=0.8),
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
for name, params in param_candidates.items():
    maes = []
    for tr_idx, val_idx in kf.split(x_train):
        X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        model = LGBMRegressor(**params, verbose=-1)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        maes.append(mean_absolute_error(y_val, pred))
    mean_mae = np.mean(maes)
    results[name] = mean_mae
    print(f"{name}: MAE = {mean_mae:.4f} (+/- {np.std(maes):.4f})")

print()
best = min(results, key=results.get)
print(f"=== 가장 좋은 조합: {best} (MAE {results[best]:.4f}) ===")

baseline(#00): MAE = 0.2117 (+/- 0.0036)
more_trees_slow_lr: MAE = 0.1884 (+/- 0.0050)
deeper: MAE = 0.2131 (+/- 0.0031)
regularized: MAE = 0.1855 (+/- 0.0054)

=== 가장 좋은 조합: regularized (MAE 0.1855) ===


## 6. Submission

In [ ]:
# regularized 조합으로 전체 데이터 학습 후 제출 파일 저장
import os

sample_submission = pd.read_csv('../data/sample_submission.csv')

best_model = LGBMRegressor(**param_candidates['regularized'], verbose=-1)
best_model.fit(x_train, y_train)
pred = best_model.predict(x_test)

os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = pred
sample_submission.to_csv('../submissions/submit_03_hyperparameter_tuning.csv', index=False)
sample_submission.head()

,ID,stress_score
0,TEST_0000,0.582107
1,TEST_0001,0.881782
2,TEST_0002,0.250853
3,TEST_0003,0.397290
4,TEST_0004,0.578843
